# CropDetector — Phase 9 / Stage B / Part 2
## Domain-balanced supported-crop recognizer (experiment, **not Android deployment**)

The Stage B Part 1 model correctly rejects only **180/303 (59.41%)** of the retrospectively reserved peach, raspberry and strawberry images. The 123 incorrect accepts make deployment unsafe. Part 2 adds **unaugmented PlantVillage** photographs of both supported and unsupported crop species to our original, group-screened PlantDoc training set. Five classes remain **Maize, Grape, Potato, Tomato, Other Crop**.

**Required user uploads:** `CropDetector_Phase9_Part2_Data.zip` and `CropDetector_Phase9_StageB_Part1_Results.zip`. This notebook does **not** need or modify the Stage A or Stage C models. GPU recommended. Outputs `CropDetector_Phase9_StageB_Part2_Results.zip`.

**Limitations:** Both sources contain publication photographs; PlantVillage images are commonly photographed against uniform backgrounds. The previous 'unseen species' test has already been inspected and influenced model-design decisions, so the Part 2 result is **retrospective diagnostic**, not a fresh independent test. New sources and real phone images are necessary before enabling Stage B.

## Cell 1 — Environment and reproducibility

In [ ]:
%pip -q install kagglehub imagehash pillow scikit-learn

import os, io, json, sys, math, shutil, subprocess, hashlib, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
import imagehash, kagglehub
from scipy.special import softmax
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from google.colab import files

SEED=42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
ROOT=Path('/content/cropdetector_stageB2'); ROOT.mkdir(parents=True, exist_ok=True)
OUT=ROOT/'results'; OUT.mkdir(exist_ok=True)
CLASSES=['Maize','Grape','Potato','Tomato','Other Crop']
OTHER=4
IMG_SIZE=160
BATCH_SIZE=32
print('Python:',sys.version.split()[0], 'TensorFlow:',tf.__version__)
print('GPU:',tf.config.list_physical_devices('GPU'))

## Cell 2 — Restore and verify the two saved ZIPs

In [ ]:
required_names=['CropDetector_Phase9_Part2_Data.zip','CropDetector_Phase9_StageB_Part1_Results.zip']
for filename in required_names:
    dest=ROOT/filename
    if not dest.exists():
        print('Upload',filename)
        up=files.upload()
        assert filename in up, f'Upload the required {filename}, not a different ZIP'
        dest.write_bytes(up[filename])

with zipfile.ZipFile(ROOT/required_names[0]) as z:
    audit=json.loads(z.read('audit_report.json'))
with zipfile.ZipFile(ROOT/required_names[1]) as z:
    v1_config=json.loads(z.read('stageB_config.json'))
    v1_metrics=json.loads(z.read('stageB_metrics.json'))
    pd_train=pd.read_csv(io.BytesIO(z.read('stageB_train_manifest.csv')))
    pd_dev=pd.read_csv(io.BytesIO(z.read('stageB_dev_manifest.csv')))
    pd_unseen=pd.read_csv(io.BytesIO(z.read('stageB_unseen_species_manifest.csv')))

assert v1_config['class_order']==CLASSES
assert v1_config['plantdoc_commit']==audit['plantdoc_git_commit']
assert len(pd_train)==1441 and len(pd_dev)==360 and len(pd_unseen)==303
for left,right in [(pd_train,pd_dev),(pd_train,pd_unseen),(pd_dev,pd_unseen)]:
    assert not set(left.Group)&set(right.Group)
    assert not set(left.SHA256)&set(right.SHA256)
print('Part 1 baseline:',v1_metrics)
print('PlantDoc commit:',audit['plantdoc_git_commit'])

## Cell 3 — Restore immutable PlantDoc snapshot and verify source hashes

In [ ]:
PLANTDOC=ROOT/'PlantDoc-Dataset'
REPO='https://github.com/pratikkayal/PlantDoc-Dataset.git'
COMMIT=audit['plantdoc_git_commit']
if not (PLANTDOC/'.git').exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(PLANTDOC)],check=True)
head=subprocess.check_output(['git','-C',str(PLANTDOC),'rev-parse','HEAD'],text=True).strip()
if head!=COMMIT:
    subprocess.run(['git','-C',str(PLANTDOC),'fetch','--depth','1','origin',COMMIT],check=True)
    subprocess.run(['git','-C',str(PLANTDOC),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(PLANTDOC),'rev-parse','HEAD'],text=True).strip()==COMMIT

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

for d in [pd_train,pd_dev,pd_unseen]:
    d['Path']=d.RelativePath.map(lambda r:str(PLANTDOC/str(r)))
    assert all(Path(x).is_file() for x in d.Path)
    bad=[r.RelativePath for r in d.itertuples(index=False) if sha256_file(r.Path)!=r.SHA256]
    assert not bad, 'PlantDoc SHA256 mismatch, example: '+str(bad[:2])
    d['Source']='PlantDoc'
    d['CropTarget']=d.CropTarget.astype(int)
print('All PlantDoc image hashes verified.')

## Cell 4 — Download **unaugmented** PlantVillage images

We use the public Kaggle mirror of PlantVillage (`emmarex/plantdisease`). Only RGB/original color images are retained if multiple variants exist. **Peach, raspberry and strawberry are excluded in their entirety** from training. We also remove exact/near duplicates of all PlantDoc images included in this Stage B experiment, including its development and diagnostic sets.

Download size may be substantial. If KaggleHub returns an access error, do **not** substitute augmented imagery or a different dataset silently; report the failure instead.

In [ ]:
pv_root=Path(kagglehub.dataset_download('emmarex/plantdisease'))
print('PlantVillage download:',pv_root)
image_ext={'.jpg','.jpeg','.png','.bmp','.webp'}
raw_pv=[]
for p in pv_root.rglob('*'):
    if not p.is_file() or p.suffix.lower() not in image_ext: continue
    folder=p.parent.name
    if '___' not in folder: continue
    parts=[x.lower() for x in p.parts]
    if any('segmented' in x or 'grayscale' in x or 'grey' in x for x in parts): continue
    species=folder.split('___',1)[0].lower().replace(' ','').replace('_','')
    if 'corn' in species or 'maize' in species: label='Maize'
    elif species.startswith('grape'): label='Grape'
    elif species.startswith('potato'): label='Potato'
    elif species.startswith('tomato'): label='Tomato'
    elif species.startswith(('apple','blueberry','cherry','orange','pepper','soybean','squash')): label='Other Crop'
    else: continue  # NOTE: peach, raspberry, strawberry deliberately excluded
    raw_pv.append((str(p),folder,label,species))

pv_all=pd.DataFrame(raw_pv,columns=['Path','Folder','CropLabel','Species'])
assert not pv_all.empty
# If both a color variant and a root variant were found, prefer color-only.
color_rows=pv_all.Path.map(lambda p:'color' in [x.lower() for x in Path(p).parts])
if color_rows.any(): pv_all=pv_all.loc[color_rows].copy()
pv_all=pv_all.drop_duplicates('Path').reset_index(drop=True)
print(pv_all.groupby(['CropLabel','Species']).size().to_string())
for cls in CLASSES:
    assert cls in pv_all.CropLabel.values, 'Missing PlantVillage category: '+cls

## Cell 5 — Stratified subset selection, hash screening, near-duplicate grouping

Sampling preserves multiple diseases/conditions within each supported species. We use caps to limit runtime and avoid overwhelming PlantDoc. We form pHash near-duplicate components at Hamming distance ≤ 4 before splitting sampled PlantVillage photos.

In [ ]:
rng=np.random.default_rng(SEED)
# Caps apply to a full species, not a disease folder. Other-crop species have smaller caps.
KEEP_KNOWN_PER_SPECIES=600
KEEP_OTHER_PER_SPECIES=150
rows=[]
for (label,species),g in pv_all.groupby(['CropLabel','Species'],sort=True):
    cap=KEEP_OTHER_PER_SPECIES if label=='Other Crop' else KEEP_KNOWN_PER_SPECIES
    per_folder={f:grp.sample(frac=1,random_state=SEED)['Path'].tolist()
                for f,grp in g.groupby('Folder',sort=True)}
    idx={f:0 for f in per_folder}
    chosen=[]
    while len(chosen)<cap:
        progressed=False
        for folder in sorted(per_folder):
            if idx[folder]<len(per_folder[folder]):
                chosen.append((per_folder[folder][idx[folder]],folder,label,species))
                idx[folder]+=1; progressed=True
                if len(chosen)==cap:break
        if not progressed:break
    rows+=chosen
selected=pd.DataFrame(rows,columns=['Path','Folder','CropLabel','Species'])
print('Selected before screening:',len(selected))

# Check image readability and compute pHash; eliminate exact duplicate files too.
meta=[]
for r in selected.itertuples(index=False):
    try:
        with Image.open(r.Path) as im:
            im=ImageOps.exif_transpose(im).convert('RGB')
            h=str(imagehash.phash(im))
        meta.append({'Path':r.Path,'Folder':r.Folder,'CropLabel':r.CropLabel,
                     'Species':r.Species,'SHA256':sha256_file(r.Path),'pHash':h})
    except Exception as e:
        print('Skipping corrupt PlantVillage image:',r.Path,str(e)[:100])
pv=pd.DataFrame(meta).drop_duplicates('SHA256').reset_index(drop=True)
assert len(pv)>1000, 'Unexpectedly small set; inspect downloaded dataset.'

pd_hashes=[int(s,16) for s in pd.concat([pd_train,pd_dev,pd_unseen]).pHash.dropna()]
keep=[]
for r in pv.itertuples(index=False):
    h=int(r.pHash,16)
    keep.append(not any((h^p).bit_count()<=4 for p in pd_hashes))
pv=pv.loc[keep].reset_index(drop=True)
print('PlantVillage images after cross-source near-duplicate screening:',len(pv))

n=len(pv);parent=list(range(n))
def find(a):
    while parent[a]!=a:
        parent[a]=parent[parent[a]];a=parent[a]
    return a
def union(a,b):
    a=find(a);b=find(b)
    if a!=b:parent[b]=a
ph=[int(x,16) for x in pv.pHash]
for i in range(n):
    for j in range(i+1,n):
        if (ph[i]^ph[j]).bit_count()<=4:union(i,j)
pv['Group']=['pv2_%05d'%find(i) for i in range(n)]
print('Near-duplicate groups:',pv.Group.nunique(),'of',len(pv),'images')
# Sanity: flag if highly similar images have conflicting crop labels.
conflicts=pv.groupby('Group').CropLabel.nunique()
conflict_groups=set(conflicts[conflicts>1].index)
if conflict_groups:
    print('Dropping conflicting near-duplicate groups:',len(conflict_groups))
    pv=pv.loc[~pv.Group.isin(conflict_groups)].copy()

# Group-disjoint 85:15 PV split within class.
pv['Partition']='train'
for label,g in pv.groupby('CropLabel',sort=True):
    groups=np.array(sorted(g.Group.unique()))
    rng.shuffle(groups)
    dev_count=max(1,round(len(groups)*0.15))
    pv.loc[g.index[g.Group.isin(set(groups[:dev_count]))],'Partition']='dev'
pv['CropTarget']=pv.CropLabel.map({s:i for i,s in enumerate(CLASSES)}).astype(int)
pv['Source']='PlantVillage'
pv_train=pv.loc[pv.Partition=='train'].copy()
pv_dev=pv.loc[pv.Partition=='dev'].copy()
assert not set(pv_train.Group)&set(pv_dev.Group)
print(pd.crosstab(pv.CropLabel,pv.Partition))

## Cell 6 — Combine PlantDoc and PlantVillage, then verify split integrity

In [ ]:
columns=['Path','Source','CropLabel','CropTarget','Folder','Group','SHA256','pHash']
train=pd.concat([pd_train[columns],pv_train[columns]],ignore_index=True)
dev=pd_dev[columns].copy().reset_index(drop=True)  # Primary field-like development set
pv_dev=pv_dev[columns].copy().reset_index(drop=True)
unseen=pd_unseen[columns].copy().reset_index(drop=True)
assert train.CropTarget.between(0,4).all()
assert not set(train.Group)&set(dev.Group)
assert not set(train.Group)&set(pv_dev.Group)
assert not set(train.Group)&set(unseen.Group)
assert not set(train.SHA256)&set(dev.SHA256)
assert not set(train.SHA256)&set(unseen.SHA256)
assert not set(pv_train.Group)&set(pv_dev.Group)
print('Training:',len(train),'PlantDoc dev:',len(dev),'PlantVillage dev:',len(pv_dev),
      'Retrospective unseen species:',len(unseen))
print(pd.crosstab(train.CropLabel,train.Source))
train.to_csv(OUT/'stageB2_train_manifest.csv',index=False)
dev.to_csv(OUT/'stageB2_plantdoc_dev_manifest.csv',index=False)
pv_dev.to_csv(OUT/'stageB2_plantvillage_dev_manifest.csv',index=False)
unseen.to_csv(OUT/'stageB2_retrospective_unseen_manifest.csv',index=False)

## Cell 7 — Balanced training pipeline

Training draws each crop class and each source equally where available. This prevents the simple source-label shortcut of putting every supported photo from one dataset and every unsupported photo from another. All paths are predecoded using PIL, with EXIF correction and bilinear RGB resize. Augmentation is training-only.

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE

def pil_preprocess(path_bytes):
    with Image.open(os.fsdecode(path_bytes)) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        return np.asarray(im.resize((IMG_SIZE,IMG_SIZE),Image.Resampling.BILINEAR),dtype=np.uint8)

def decode(path,y):
    x=tf.numpy_function(pil_preprocess,[path],tf.uint8)
    x.set_shape((IMG_SIZE,IMG_SIZE,3))
    return tf.cast(x,tf.float32),tf.cast(y,tf.int32)

def augment(x,y):
    x=tf.image.random_flip_left_right(x)
    x=tf.image.random_flip_up_down(x)
    x=tf.image.random_brightness(x,24.0)
    x=tf.image.random_contrast(x,0.8,1.2)
    x=tf.clip_by_value(x,0.0,255.0)
    return x,y

streams=[];names=[]
for label in CLASSES:
    for source in ['PlantDoc','PlantVillage']:
        part=train.loc[train.CropLabel.eq(label)&train.Source.eq(source)]
        if len(part)==0:continue
        ds=tf.data.Dataset.from_tensor_slices((part.Path.to_numpy(dtype=str),
                 part.CropTarget.to_numpy(dtype=np.int32)))
        ds=ds.shuffle(len(part),seed=SEED,reshuffle_each_iteration=True).repeat()
        ds=ds.map(decode,num_parallel_calls=AUTOTUNE)
        ds=ds.map(augment,num_parallel_calls=AUTOTUNE)
        streams.append(ds);names.append((label,source,len(part)))
assert len(streams)==10, f'Missing class/source pair: {names}'
train_ds=tf.data.Dataset.sample_from_datasets(streams,weights=[1/len(streams)]*len(streams),seed=SEED)
train_ds=train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def make_eval(frame):
    ds=tf.data.Dataset.from_tensor_slices((frame.Path.to_numpy(dtype=str),
            frame.CropTarget.to_numpy(dtype=np.int32)))
    return ds.map(decode,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
dev_ds=make_eval(dev)
pv_dev_ds=make_eval(pv_dev)
unseen_ds=make_eval(unseen)
STEPS_PER_EPOCH=max(75,math.ceil(len(train)/BATCH_SIZE))
print('Streams:',names,'steps/epoch:',STEPS_PER_EPOCH)

## Cell 8 — MobileNetV2-0.5, ImageNet initialization and classification head training

In [ ]:
from tensorflow.keras import layers
base=tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE,IMG_SIZE,3),alpha=0.5,
                                       include_top=False,weights='imagenet')
base.trainable=False
inp=layers.Input(shape=(IMG_SIZE,IMG_SIZE,3),dtype=tf.float32,name='rgb_image')
x=layers.Rescaling(1./127.5,offset=-1.0,name='mobilenet_preprocess')(inp)
x=base(x,training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dropout(0.3)(x)
logits=layers.Dense(5,name='crop_logits')(x)
model=tf.keras.Model(inp,logits,name='StageB2_CropRecognizer')

def compile_model(m,lr):
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
       loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
       metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
compile_model(model,5e-4)
HEAD=OUT/'best_stageB2_head.keras'
cbs=[tf.keras.callbacks.ModelCheckpoint(str(HEAD),monitor='val_loss',save_best_only=True),
     tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True),
     tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=2,factor=0.5,min_lr=1e-5)]
h_head=model.fit(train_ds,validation_data=dev_ds,steps_per_epoch=STEPS_PER_EPOCH,
                 epochs=12,callbacks=cbs)
print('Frozen head:',HEAD)

## Cell 9 — Fine-tune the last 35 non-BatchNorm backbone layers

In [ ]:
ft=tf.keras.models.load_model(HEAD,compile=False)
ft_base=next(x for x in ft.layers if isinstance(x,tf.keras.Model))
ft_base.trainable=True
for layer in ft_base.layers:layer.trainable=False
for layer in ft_base.layers[-35:]:
    if not isinstance(layer,tf.keras.layers.BatchNormalization):layer.trainable=True
compile_model(ft,1e-5)
FINETUNED=OUT/'best_stageB2_finetuned.keras'
cbs2=[tf.keras.callbacks.ModelCheckpoint(str(FINETUNED),monitor='val_loss',save_best_only=True),
      tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=2,restore_best_weights=True),
      tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=2,factor=0.5,min_lr=1e-6)]
h_ft=ft.fit(train_ds,validation_data=dev_ds,steps_per_epoch=STEPS_PER_EPOCH,
            epochs=8,callbacks=cbs2)
print('Fine-tuned:',FINETUNED)

## Cell 10 — Select checkpoint and gate threshold using **PlantDoc development only**

The acceptance threshold is set to preserve at least 75% of **correctly identified and accepted** supported-crop development images, where feasible. This is stricter than counting any supported species as accepted, because misrouting a tomato leaf into the grape-specific disease pathway is a serious failure. If the minimum is infeasible, the report flags that Stage B cannot be deployed. The retrospective peach/raspberry/strawberry diagnostic does **not** influence checkpoint or threshold selection.

In [ ]:
def predict_logits(m,dataset):
    return np.asarray(m.predict(dataset.map(lambda x,y:x),verbose=0),dtype=np.float64)

def dev_summary(logits,labels):
    probs=softmax(logits,axis=1)
    pred=probs.argmax(axis=1)
    nll=-np.log(np.clip(probs[np.arange(len(labels)),labels],1e-12,1)).mean()
    return float(nll),float(accuracy_score(labels,pred)),float(balanced_accuracy_score(labels,pred))

y=dev.CropTarget.to_numpy(dtype=int)
results=[]
for label,path in [('Frozen head',HEAD),('Fine-tuned',FINETUNED)]:
    m=tf.keras.models.load_model(path,compile=False)
    logits=predict_logits(m,dev_ds)
    nll,acc,balanced=dev_summary(logits,y)
    results.append({'Checkpoint':label,'Path':str(path),'Development NLL':nll,
                    'Development accuracy':acc,'Development balanced accuracy':balanced})
comparison=pd.DataFrame(results).sort_values('Development NLL')
comparison.to_csv(OUT/'stageB2_checkpoint_comparison.csv',index=False)
display(comparison)
chosen=comparison.iloc[0]
BEST=Path(chosen.Path)
BEST_MODEL=tf.keras.models.load_model(BEST,compile=False)
prob=softmax(predict_logits(BEST_MODEL,dev_ds),axis=1)
pred=prob.argmax(axis=1)
known=(y!=OTHER)
correct_known=(pred==y)&known
known_conf=prob[:,:4].max(axis=1)
candidates=np.unique(np.r_[0.0,known_conf[known]])
TARGET_CORRECT_RETENTION=0.75
feasible=[]
for t in candidates:
    accept=(pred!=OTHER)&(known_conf>t)
    correct_rate=float(np.mean((accept&correct_known)[known]))
    if correct_rate>=TARGET_CORRECT_RETENTION:feasible.append(float(t))
threshold=max(feasible) if feasible else 0.0
if not feasible:print('WARNING: correct known-crop retention target infeasible. DO NOT DEPLOY.')
print('Selected model:',chosen.Checkpoint,'Threshold:',threshold)

def metrics_by_frame(frame,probs,t):
    labels=frame.CropTarget.to_numpy(dtype=int)
    p=probs.argmax(axis=1)
    conf=probs[:,:4].max(axis=1)
    accepted=(p!=OTHER)&(conf>t)
    known=(labels!=OTHER)
    other=~known
    out={'N':len(labels),'accuracy':float(np.mean(p==labels)),
         'balanced_accuracy':float(balanced_accuracy_score(labels,p)),
         'threshold':float(t)}
    if known.sum():
        out['known_gate_retention']=float(accepted[known].mean())
        out['known_correct_and_accepted']=float(((p==labels)&accepted)[known].mean())
        out['known_wrong_crop_and_accepted']=float(((p!=labels)&accepted)[known].mean())
    if other.sum():
        out['other_rejection']=float((~accepted[other]).mean())
        out['other_false_supported_acceptance']=float(accepted[other].mean())
    return out,p,accepted

pd_metrics,pd_pred,pd_accept=metrics_by_frame(dev,prob,threshold)
print('PlantDoc development:',json.dumps(pd_metrics,indent=2))

## Cell 11 — External-source and retrospective unseen-species diagnostics (no model selection)

In [ ]:
prob_pv=softmax(predict_logits(BEST_MODEL,pv_dev_ds),axis=1)
prob_unseen=softmax(predict_logits(BEST_MODEL,unseen_ds),axis=1)
pv_metrics,pv_pred,pv_accept=metrics_by_frame(pv_dev,prob_pv,threshold)
unseen_metrics,unseen_pred,unseen_accept=metrics_by_frame(unseen,prob_unseen,threshold)

print('PlantVillage development:',json.dumps(pv_metrics,indent=2))
print('Retrospective unseen-species diagnostic:',json.dumps(unseen_metrics,indent=2))

# Export per-image predictions for targeted error analysis.
def save_scores(frame,probs,pred,accepted,name):
    res=frame[['Path','Folder','Source','CropLabel','CropTarget','Group']].copy()
    for i,c in enumerate(CLASSES):res['Prob_'+c.replace(' ','_')]=probs[:,i]
    res['Predicted']=[CLASSES[i] for i in pred]
    res['AcceptedAsSupported']=accepted
    res['CorrectCropAndAccepted']=(frame.CropTarget.to_numpy()==pred)&accepted
    res.to_csv(OUT/name,index=False)
    return res
pds=save_scores(dev,prob,pd_pred,pd_accept,'stageB2_plantdoc_dev_scores.csv')
pvs=save_scores(pv_dev,prob_pv,pv_pred,pv_accept,'stageB2_plantvillage_dev_scores.csv')
uns=save_scores(unseen,prob_unseen,unseen_pred,unseen_accept,'stageB2_unseen_crop_scores.csv')

print('PlantDoc confusion:',pd.crosstab(pds.CropLabel,pds.Predicted),sep='\n')
print('Retrospective unseen by folder:')
print(uns.groupby('Folder').AcceptedAsSupported.agg(['size','sum','mean']))

# Records for report, but not automatic production approval.
metrics={'phase':'Phase9_StageB_Part2','selected_checkpoint':str(chosen.Checkpoint),
         'plantdoc_dev':pd_metrics,'plantvillage_dev':pv_metrics,
         'retrospective_unseen_species':unseen_metrics,
         'part1_baseline':v1_metrics,
         'pass_plantdoc_known_correct_75pct':pd_metrics.get('known_correct_and_accepted',0)>=0.75,
         'pass_plantdoc_other_rejection_85pct':pd_metrics.get('other_rejection',0)>=0.85,
         'pass_retrospective_unseen_rejection_85pct':unseen_metrics.get('other_rejection',0)>=0.85}
(OUT/'stageB2_metrics.json').write_text(json.dumps(metrics,indent=2))
print('Provisional criteria (not proof of field performance):',
      {k:v for k,v in metrics.items() if k.startswith('pass_')})

## Cell 12 — Save model, provenance, audit and reproducibility metadata

Do not integrate Stage B into Android automatically. Review recognition and rejection errors, then obtain a **new species-disjoint and device-camera test set** before deployment. `Stage A` and the 15-class Stage C stay unchanged.

In [ ]:
FINAL=OUT/'crop_detector_stageB2.keras'
shutil.copy2(BEST,FINAL)
config={'phase':'Phase9_StageB_Part2',
        'task':'supported_crop_recognition_experimental',
        'class_order':CLASSES,
        'other_class_id':OTHER,
        'input_shape':[1,IMG_SIZE,IMG_SIZE,3],
        'input_range':[0,255],
        'preprocessing':'PIL EXIF transpose, RGB, bilinear 160x160; internal MobileNetV2 [-1,1]',
        'output':'five uncalibrated logits',
        'gate':'top class must be one of four known crops and its softmax > threshold',
        'threshold':float(threshold),
        'threshold_selected_from':'PlantDoc development only',
        'model_architecture':'MobileNetV2 alpha=0.5 ImageNet pretrained',
        'model_filename':FINAL.name,
        'model_sha256':sha256_file(FINAL),
        'plantdoc_git_commit':COMMIT,
        'plantvillage_source':'kagglehub:emmarex/plantdisease',
        'part1_baseline_unseen_false_acceptance':v1_metrics['unseen_other_false_supported_acceptance'],
        'part1_species_used_as_retrospective_diagnostic':['Peach','Raspberry','Strawberry'],
        'plantvillage_train_images':len(pv_train),
        'plantvillage_dev_images':len(pv_dev),
        'plantdoc_train_images':len(pd_train),
        'plantdoc_dev_images':len(pd_dev),
        'limitations':['The retrospective unseen-species set has been inspected in Part 1.',
                       'PlantVillage is controlled-background source; not a substitute for phone photos.',
                       'PlantDoc development selected checkpoint and threshold.',
                       'No genuine unseen external test for final Android acceptance.',
                       'Further quantization and latency measurement required.']}
(OUT/'stageB2_config.json').write_text(json.dumps(config,indent=2))
print('Saved',FINAL,FINAL.stat().st_size,'bytes')

## Cell 13 — Export the results ZIP for ChatGPT review

In [ ]:
archive=shutil.make_archive(str(ROOT/'CropDetector_Phase9_StageB_Part2_Results'),'zip',OUT)
print('Exported:',archive)
with zipfile.ZipFile(archive) as z:
    for entry in z.infolist():print(entry.filename,round(entry.file_size/1024,1),'KiB')
files.download(archive)